In [1]:
def print_board(state):
    """Displays the 3x3 puzzle board."""
    for i in range(0, 9, 3):
        print(state[i:i+3])
    print()

def get_neighbors(state):
    """Generates valid next states by moving the blank tile (0)."""
    neighbors = []
    zero_idx = state.index(0)
    x, y = divmod(zero_idx, 3)


    for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            n_idx = nx * 3 + ny
            lst = list(state)
            lst[zero_idx], lst[n_idx] = lst[n_idx], lst[zero_idx]
            neighbors.append(tuple(lst))

    return neighbors

def depth_limited_search(state, goal, depth, path):
    """Performs recursive Depth-Limited Search."""
    if state == goal:
        return path
    if depth <= 0:
        return None

    for neighbor in get_neighbors(state):
        if neighbor not in path:
            result = depth_limited_search(neighbor, goal, depth - 1, path + [neighbor])
            if result:
                return result

    return None

def iterative_deepening_search(start, goal, max_depth=30):
    """Applies Iterative Deepening Search (IDS) by increasing depth limit."""
    for depth in range(max_depth + 1):
        result = depth_limited_search(start, goal, depth, [start])
        if result:
            return result, depth
    return None, -1

# Define Start and Goal States (0 represents the blank tile)
start_state = (2,8,3,1,6,4,7,0,5)
goal_state = (1,2,3,8,0,4,7,6,5)

print("Initial Board:")
print_board(start_state)

solution_path, final_depth = iterative_deepening_search(start_state, goal_state)

if solution_path:
    print(f"Solution found at depth limit: {final_depth}")
    print(f"Total moves: {len(solution_path) - 1}\n")
    print("Step-by-step moves:")
    for step, board in enumerate(solution_path):
        print(f"Step {step}:")
        print_board(board)
else:
    print("No solution found within depth limit.")


Initial Board:
(2, 8, 3)
(1, 6, 4)
(7, 0, 5)

Solution found at depth limit: 5
Total moves: 5

Step-by-step moves:
Step 0:
(2, 8, 3)
(1, 6, 4)
(7, 0, 5)

Step 1:
(2, 8, 3)
(1, 0, 4)
(7, 6, 5)

Step 2:
(2, 0, 3)
(1, 8, 4)
(7, 6, 5)

Step 3:
(0, 2, 3)
(1, 8, 4)
(7, 6, 5)

Step 4:
(1, 2, 3)
(0, 8, 4)
(7, 6, 5)

Step 5:
(1, 2, 3)
(8, 0, 4)
(7, 6, 5)



In [2]:
import sys
sys.setrecursionlimit(2000)

class EightPuzzleIDDFS:
    def __init__(self, initial_grid, goal_grid=None):
        self.initial_grid = tuple(tuple(row) for row in initial_grid)
        if goal_grid is None:
            self.goal_grid = ((1, 2, 3), (8, 0, 4), (7, 6, 5))
        else:
            self.goal_grid = tuple(tuple(row) for row in goal_grid)

        self.visited = set()
        self.solution_path = []
        self.states_path = []  # Tracks the grid state at each step

    def find_blank(self, grid):
        for r in range(3):
            for c in range(3):
                if grid[r][c] == 0:
                    return r, c

    def get_neighbors(self, grid):
        r, c = self.find_blank(grid)
        neighbors = []
        moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

        for dr, dc, move_name in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                new_grid = [list(row) for row in grid]
                new_grid[r][c], new_grid[nr][nc] = new_grid[nr][nc], new_grid[r][c]
                neighbors.append((tuple(tuple(row) for row in new_grid), move_name))
        return neighbors

    def solve(self, max_limit=50):
        for depth_limit in range(max_limit + 1):
            self.visited.clear()
            self.solution_path = []
            self.states_path = [self.initial_grid]  # Start tracking with the initial state
            if self._dfs(self.initial_grid, 0, depth_limit):
                return self.solution_path, self.states_path
        return None, None

    def _dfs(self, current_grid, depth, depth_limit):
        # 1. Visited Check FIRST
        if current_grid in self.visited:
            return False

        # 2. Goal Check SECOND
        if current_grid == self.goal_grid:
            return True

        if depth > depth_limit:
            return False

        self.visited.add(current_grid)

        for neighbor_grid, move in self.get_neighbors(current_grid):
            if neighbor_grid not in self.visited:
                self.solution_path.append(move)
                self.states_path.append(neighbor_grid)

                if self._dfs(neighbor_grid, depth + 1, depth_limit):
                    return True

                self.solution_path.pop()  # Backtrack move
                self.states_path.pop()    # Backtrack state

        self.visited.remove(current_grid)
        return False

def print_grid(grid):
    """Helper function to print a clean 3x3 layout."""
    for row in grid:
        print("  " + " ".join(str(x) if x != 0 else "_" for x in row))
    print()


start_grid = [
    [2, 8, 3],
[1, 6, 4],
[7, 0, 5]
]

solver = EightPuzzleIDDFS(start_grid)
path, states = solver.solve()

if path and states:
    print(f"--- Initial State ---")
    print_grid(states[0])

    for i in range(len(path)):
        print(f"Step {i+1}: Move {path[i]}")
        print_grid(states[i+1])

    print(f"Solution Found in {len(path)} steps! (Optimal)")
else:
    print("\nNo solution found.")

--- Initial State ---
  2 8 3
  1 6 4
  7 _ 5

Step 1: Move Up
  2 8 3
  1 _ 4
  7 6 5

Step 2: Move Up
  2 _ 3
  1 8 4
  7 6 5

Step 3: Move Left
  _ 2 3
  1 8 4
  7 6 5

Step 4: Move Down
  1 2 3
  _ 8 4
  7 6 5

Step 5: Move Right
  1 2 3
  8 _ 4
  7 6 5

Solution Found in 5 steps! (Optimal)
